# IEEE-CIS SCARF Fusion Benchmark on Colab

Notebook nay dung de benchmark `SCARF score fusion` theo kieu `baseline Nx3` vs `fused Nx4` tren Google Colab truoc khi can nhac tich hop vao notebook IEEE chinh.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
GITHUB_REPO = "https://github.com/Thanh-000/FDB1-DoAn.git"
REPO_PARENT = "/content/drive/MyDrive"
REPO_NAME = "FDB1-DoAn"
REPO_DIR = f"{REPO_PARENT}/{REPO_NAME}"

ZIP_PATH = "/content/drive/MyDrive/MVS_XAI_Data/ieee-fraud-detection.zip"
EXTRACT_DIR = "/content/ieee-fraud-detection"

FOLD_INDEX = 0
N_SPLITS = 5
INNER_SPLITS = 2
PRETRAIN_EPOCHS = 10
DOWNSTREAM_EPOCHS = 10
BATCH_SIZE = 2048
HIDDEN_DIM = 256
EMB_DIM = 128
PROJ_DIM = 128
LR = 1e-3
CORRUPTION_RATE = 0.6
NO_PRETRAIN = False

In [ ]:
import os
if os.path.exists(REPO_DIR):
    %cd "$REPO_DIR"
    !git pull
else:
    %cd "$REPO_PARENT"
    !git clone "$GITHUB_REPO" "$REPO_NAME"
    %cd "$REPO_DIR"

In [ ]:
import os, zipfile, glob

if not os.path.exists(EXTRACT_DIR):
    os.makedirs(EXTRACT_DIR, exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
        zip_ref.extractall(EXTRACT_DIR)

txn_files = glob.glob(f"{EXTRACT_DIR}/**/train_transaction.csv", recursive=True)
id_files = glob.glob(f"{EXTRACT_DIR}/**/train_identity.csv", recursive=True)
print('txn:', txn_files[:1])
print('id :', id_files[:1])

DATA_DIR = os.path.dirname(txn_files[0])
print('DATA_DIR =', DATA_DIR)

In [ ]:
%cd "$REPO_DIR"
!pip -q install scikit-learn xgboost lightgbm catboost torch

In [ ]:
import torch, sklearn, xgboost, lightgbm, catboost
print('torch:', torch.__version__)
print('cuda:', torch.cuda.is_available())
print('gpu :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO CUDA')
print('sklearn:', sklearn.__version__)
print('xgboost:', xgboost.__version__)
print('lightgbm:', lightgbm.__version__)
print('catboost:', catboost.__version__)

In [ ]:
cmd = [
    'python', 'run_ieee_scarf_fusion.py',
    '--data-dir', DATA_DIR,
    '--fold-index', str(FOLD_INDEX),
    '--n-splits', str(N_SPLITS),
    '--inner-splits', str(INNER_SPLITS),
    '--pretrain-epochs', str(PRETRAIN_EPOCHS),
    '--downstream-epochs', str(DOWNSTREAM_EPOCHS),
    '--batch-size', str(BATCH_SIZE),
    '--hidden-dim', str(HIDDEN_DIM),
    '--emb-dim', str(EMB_DIM),
    '--proj-dim', str(PROJ_DIM),
    '--lr', str(LR),
    '--corruption-rate', str(CORRUPTION_RATE),
]
if NO_PRETRAIN:
    cmd.append('--no-pretrain')
print('Running:')
print(' '.join(cmd))

In [ ]:
import subprocess
result = subprocess.run(cmd, text=True, capture_output=True)
print(result.stdout)
print(result.stderr)
print('returncode =', result.returncode)